# Phase 6: Final Held-Out Test Evaluation

This notebook loads the persisted frozen Phase 6 evaluation. It does not rerun training, tune models, or alter the selected specifications.

## Frozen model selection

The selection rule was lowest validation MAE, then lowest validation RMSE, then highest validation R2. The frozen selections are Extended Ridge(alpha=1.0) at 60s and Extended Linear Regression at 300s. Test results were not available during selection.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
results_dir = project_root / 'results' / 'freetwinev'
figures_dir = results_dir / 'figures'
pooled = pd.read_csv(results_dir / 'phase6_results.csv')
per_run = pd.read_csv(results_dir / 'phase6_results_per_run.csv')
predictions = pd.read_csv(results_dir / 'phase6_predictions_test.csv')
audit = json.loads((results_dir / 'phase6_audit.json').read_text(encoding='utf-8'))
print('Loaded persisted Phase 6 results; audit:', audit['overall_status'])

## Pooled held-out test metrics

These metrics are computed over ID18 and ID19 only. Absolute-temperature metrics use T_future_pred = T_cell_max + predicted_DeltaT.

In [ ]:
pooled

## Per-run held-out metrics

ID18 and ID19 are shown separately so regime-specific differences are visible rather than hidden by pooled metrics.

In [ ]:
per_run

## Validation versus test comparison

This is a generalization comparison only. It is not used to revise the frozen model selection.

In [ ]:
validation = pd.DataFrame([
    {'Horizon': '60s', 'Model': 'Ridge Regression', 'Val_MAE': 0.1598, 'Val_RMSE': 0.7228, 'Val_R2': 0.7890},
    {'Horizon': '300s', 'Model': 'Linear Regression', 'Val_MAE': 2.2582, 'Val_RMSE': 4.5396, 'Val_R2': 0.5027},
])
comparison = validation.merge(pooled[['Horizon', 'Test_MAE', 'Test_RMSE', 'Test_R2']], on='Horizon')
comparison['MAE_Change'] = comparison['Test_MAE'] - comparison['Val_MAE']
comparison['RMSE_Change'] = comparison['Test_RMSE'] - comparison['Val_RMSE']
comparison['R2_Change'] = comparison['Test_R2'] - comparison['Val_R2']
comparison

## Diagnostic figures

The runner generated five held-out test figures: actual versus predicted DeltaT, residual distributions, residuals versus target, absolute-temperature tracking, and per-run MAE.

In [ ]:
figure_names = [
    '31_phase6_actual_vs_predicted_deltaT.png',
    '32_phase6_residual_distributions.png',
    '33_phase6_residuals_vs_target.png',
    '34_phase6_absolute_temperature_tracking.png',
    '35_phase6_per_run_mae.png',
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, name in zip(axes.flat, figure_names):
    ax.imshow(plt.imread(figures_dir / name))
    ax.set_title(name)
    ax.axis('off')
axes.flat[-1].axis('off')
plt.tight_layout()

## Integrity audit

The audit verifies exact test membership, frozen preprocessing and specifications, 21 features, no test tuning, prediction counts, no missing predictions, and no duplicate prediction rows.

In [ ]:
assert audit['overall_status'] == 'PASS'
assert audit['protocol_check']['test_runs'] == ['ID18', 'ID19']
assert audit['feature_check']['feature_count'] == 21
assert audit['test_metrics_used_for_selection'] is False
assert audit['post_test_tuning'] is False
audit['post_evaluation_checks']

## Conclusion

At 60s, the frozen Ridge model produced pooled test MAE 0.2271 C, RMSE 0.8256 C, and R2 0.8060. At 300s, the frozen OLS model produced pooled test MAE 1.9302 C, RMSE 4.9649 C, and R2 0.5836. These are final held-out results for this experiment. No post-test tuning or Phase 7 work was performed.